# 41 — Intraday và dòng lệnh

Mở đầu Track 4. Dữ liệu trong phiên có bốn bề mặt, và ba trong số đó mang cái
tên nghe giống nhau — "chủ động" — nhưng đo ba thứ khác nhau. Notebook này đo
chính xác chúng khác nhau ở đâu, bằng phép cộng.

1. `ticks()` — từng lệnh khớp, và **ba** giá trị của `side`
2. `net_active_value()` — ba rổ, tính bằng VND
3. `active_volume()` — hai rổ, tính bằng khối lượng · **và nó làm gì với ATO/ATC**
4. `supply_demand()` — lệnh **đặt**, không phải lệnh khớp

Cộng thêm ràng buộc thực tế quan trọng nhất của Track này: `max_intraday_days`.

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, hom_nay, lui_ngay, ty_dong
from finlens_examples.charts import CHUOI, GIAM, TANG, THAM_CHIEU

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

MA = "HPG"

## 1 · Chọn một phiên có thật — trên **mọi** bề mặt sẽ dùng

Đừng viết cứng ngày, và đừng lấy phiên gần nhất của một bề mặt duy nhất. Bốn
bảng dưới đây **không cùng độ trễ**: giá về ngay sau phiên, còn sổ lệnh đặt
thường về chậm hơn một phiên.

Phiên phân tích phải là phiên gần nhất **có mặt ở tất cả** — nếu không, một
cell nào đó ở giữa notebook sẽ nhận về bảng rỗng và ném `IndexError`.

In [2]:
BE_MAT = {
    "eod.ohlcv": client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, ngay=20)),
    "eod.supply_demand": client.eod.stock.supply_demand(MA, start=lui_ngay(HOM_NAY, ngay=20)),
    "eod.active_volume": client.eod.stock.active_volume(MA, start=lui_ngay(HOM_NAY, ngay=20)),
    "eod.investor.flow": client.eod.stock.investor.flow(MA, start=lui_ngay(HOM_NAY, ngay=20)),
}

print("Phiên gần nhất của từng bề mặt:")
for ten, d in BE_MAT.items():
    print(f"  {ten:<20} {d['date'].max():%d/%m/%Y}")

PHIEN = min(d["date"].max() for d in BE_MAT.values())
print(f"\n→ Phiên phân tích (giao của cả bốn): {PHIEN:%d/%m/%Y}")

gan_day = BE_MAT["eod.ohlcv"]
KL_EOD = float(gan_day.loc[gan_day["date"] == PHIEN, "volume"].iloc[0])
print(f"Khối lượng khớp cả phiên: {KL_EOD:,.0f} cổ phiếu")

Phiên gần nhất của từng bề mặt:
  eod.ohlcv            11/08/2026
  eod.supply_demand    10/08/2026
  eod.active_volume    11/08/2026
  eod.investor.flow    11/08/2026

→ Phiên phân tích (giao của cả bốn): 10/08/2026
Khối lượng khớp cả phiên: 16,732,700 cổ phiếu


## 2 · `ticks()` — từng lệnh khớp

Một mã, một phiên mỗi lời gọi. Một phiên phái sinh sôi động là hơn 90.000 dòng.

In [3]:
tick = client.intraday.stock.ticks(MA, date=PHIEN.strftime("%Y-%m-%d"))

print(f"{len(tick):,} lệnh khớp · đơn vị: {tick.attrs['finlens']['units']}")
tick.head(4)

3,501 lệnh khớp · đơn vị: {'symbol': None, 'time': None, 'price': 'kVND', 'volume': 'share', 'side': None, 'value': 'VND'}


,symbol,time,price,volume,side,value
0,HPG,2026-08-10 09:15:15+07:00,22.30,190900.0,auction,4.257070e+09
1,HPG,2026-08-10 09:15:29+07:00,22.25,3000.0,auction,6.675000e+07
2,HPG,2026-08-10 09:15:33+07:00,22.25,200.0,auction,4.450000e+06
3,HPG,2026-08-10 09:15:37+07:00,22.25,200.0,auction,4.450000e+06


### Phép kiểm đầu tiên: tick có cộng lại bằng EOD không?

Chạy phép này một lần với mỗi nguồn tick mới. Nếu nó không khớp thì mọi thứ
bạn tính từ tick đều đáng ngờ.

In [4]:
kl_tick = tick["volume"].sum()
print(f"Tổng khối lượng từ tick : {kl_tick:>14,.0f}")
print(f"Khối lượng từ EOD       : {KL_EOD:>14,.0f}")
print(f"Lệch                    : {kl_tick - KL_EOD:>14,.0f}  ({abs(kl_tick - KL_EOD) / KL_EOD:.6%})")

Tổng khối lượng từ tick :     16,732,700
Khối lượng từ EOD       :     16,732,700
Lệch                    :              0  (0.000000%)


⚠️ Lưu ý giá trong tick là **giá thô**, chưa điều chỉnh quyền — khác với EOD
mặc định `adjusted=True`. Khối lượng thì khớp tuyệt đối, còn giá thì chỉ khớp
khi mã đó chưa có sự kiện quyền nào sau phiên này.

## 3 · ⚠️ `side` có BA giá trị, không phải hai

`buy` — bên mua nâng giá chạm bên bán. `sell` — bên bán hạ giá chạm bên mua.
**`auction`** — khớp lệnh định kỳ ATO/ATC.

Phiên định kỳ **không có bên chủ động**: mọi lệnh khớp cùng một giá tại cùng
một thời điểm. Xếp nó vào mua hay bán đều sai.

In [5]:
theo_side = tick.groupby("side", observed=True).agg(
    so_lenh=("volume", "count"),
    khoi_luong=("volume", "sum"),
    gia_tri=("value", "sum"),
)
theo_side["% khối lượng"] = (theo_side["khoi_luong"] / kl_tick * 100).round(2)
theo_side

,so_lenh,khoi_luong,gia_tri,% khối lượng
side,,,,
auction,12,1167500.0,2.584126e+10,6.98
buy,1332,7233400.0,1.600849e+11,43.23
sell,2157,8331800.0,1.839562e+11,49.79


Phiên định kỳ chiếm hai chữ số phần trăm khối lượng ở nhiều mã — quá lớn để
giấu vào một trong hai rổ còn lại. Đó là lý do `side` có ba giá trị chứ không
phải hai.

In [6]:
ve_side = theo_side.reset_index().assign(
    nhan=lambda d: d["side"].map({"buy": "Mua chủ động", "sell": "Bán chủ động", "auction": "Khớp định kỳ (ATO/ATC)"})
)

fig = go.Figure(
    go.Bar(
        x=ve_side["nhan"],
        y=ve_side["% khối lượng"],
        marker=dict(
            color=[TANG if s == "buy" else GIAM if s == "sell" else THAM_CHIEU for s in ve_side["side"]],
            line=dict(color="#fcfcfb", width=2),
        ),
        text=[f"{v:.1f}%" for v in ve_side["% khối lượng"]],
        textposition="outside",
        textfont=dict(color="#52514e", size=12),
        cliponaxis=False,
        hovertemplate="%{x}: %{y:.2f}%<extra></extra>",
    )
)
fig.update_layout(
    title_text=f"{MA} — cơ cấu khối lượng theo bên chủ động, phiên {PHIEN:%d/%m/%Y}<br>"
    "<sub style='color:#52514e'>Vàng là phiên định kỳ: không có bên nào chủ động, nên nó là rổ thứ ba</sub>",
    yaxis_title="% khối lượng phiên",
    height=440,
    showlegend=False,
)
fig

## 4 · VWAP và diễn biến trong phiên

VWAP là giá bình quân gia quyền khối lượng. Tính từ tick thì nó chính xác tới
từng lệnh, không phải xấp xỉ từ nến.

In [7]:
tick = tick.sort_values("time")
tick["kl_luy_ke"] = tick["volume"].cumsum()
tick["gt_luy_ke"] = (tick["price"] * tick["volume"]).cumsum()
tick["vwap"] = tick["gt_luy_ke"] / tick["kl_luy_ke"]

vwap_phien = tick["gt_luy_ke"].iloc[-1] / tick["kl_luy_ke"].iloc[-1]
print(f"VWAP cả phiên: {vwap_phien:,.3f} nghìn VND")
gia_dong_cua = float(gan_day.loc[gan_day["date"] == PHIEN, "close"].iloc[0])
print(f"Giá đóng cửa : {gia_dong_cua:,.3f} nghìn VND (đã điều chỉnh)")
print(f"Giá khớp cuối: {tick['price'].iloc[-1]:,.3f} nghìn VND (thô)")

VWAP cả phiên: 22.105 nghìn VND
Giá đóng cửa : 22.100 nghìn VND (đã điều chỉnh)
Giá khớp cuối: 22.100 nghìn VND (thô)


In [8]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.68, 0.32])

fig.add_trace(
    go.Scatter(x=tick["time"], y=tick["price"], name="Giá khớp", mode="lines",
               line=dict(width=1, color="#c3c2b7"), hovertemplate="%{y:.2f}<extra>giá khớp</extra>"),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=tick["time"], y=tick["vwap"], name="VWAP luỹ kế", mode="lines",
               line=dict(width=2, color=CHUOI[0]), hovertemplate="%{y:.3f}<extra>VWAP</extra>"),
    row=1, col=1,
)

phut = (
    tick.assign(phut=tick["time"].dt.floor("5min"))
    .groupby(["phut", "side"], observed=True)["volume"]
    .sum()
    .unstack(fill_value=0)
)
for cot, mau, nhan in [("buy", TANG, "Mua chủ động"), ("sell", GIAM, "Bán chủ động"), ("auction", THAM_CHIEU, "Định kỳ")]:
    if cot in phut.columns:
        fig.add_trace(
            go.Bar(x=phut.index, y=phut[cot], name=nhan, marker=dict(color=mau, line=dict(width=0))),
            row=2, col=1,
        )

fig.update_layout(
    barmode="stack",
    title_text=f"{MA} — giá, VWAP và dòng lệnh theo 5 phút · {PHIEN:%d/%m/%Y}",
    height=680,
    hovermode="x unified",
)
fig.update_yaxes(title_text="nghìn VND", row=1, col=1)
fig.update_yaxes(title_text="cổ phiếu / 5 phút", row=2, col=1)
fig

## 5 · ⚠️ Ba đại lượng "chủ động" — và phép cộng phân biệt chúng

Đây là phần quan trọng nhất của notebook. Ba bề mặt cùng nói về "chủ động":

In [9]:
ngay_str = PHIEN.strftime("%Y-%m-%d")

nav = client.intraday.stock.net_active_value(MA, interval="1h", start=ngay_str, end=ngay_str)
av = client.eod.stock.active_volume(MA, start=ngay_str, end=ngay_str)
sd = client.eod.stock.supply_demand(MA, start=ngay_str, end=ngay_str)

print("net_active_value — BA rổ, có cả cột tiền:")
print(f"  {list(nav.columns)}\n")
print("active_volume — HAI rổ, KHÔNG có cột tiền nào:")
print(f"  {list(av.columns)}\n")
print("supply_demand — lệnh ĐẶT, mọi cột đều mang chữ _order_:")
print(f"  {list(sd.columns)}")

net_active_value — BA rổ, có cả cột tiền:
  ['symbol', 'time', 'buy_value', 'sell_value', 'net_value', 'auction_value', 'buy_volume', 'sell_volume', 'net_volume', 'auction_volume']

active_volume — HAI rổ, KHÔNG có cột tiền nào:
  ['symbol', 'date', 'active_buy_volume', 'active_sell_volume', 'net_active_volume']

supply_demand — lệnh ĐẶT, mọi cột đều mang chữ _order_:
  ['symbol', 'date', 'buy_order_volume', 'sell_order_volume', 'net_order_volume', 'buy_order_count', 'sell_order_count', 'net_order_count']


In [10]:
nav_tong = nav[["buy_volume", "sell_volume", "auction_volume"]].sum()
av_tong = av[["active_buy_volume", "active_sell_volume"]].iloc[0]

so_sanh = pd.DataFrame(
    {
        "net_active_value": [nav_tong["buy_volume"], nav_tong["sell_volume"], nav_tong["auction_volume"]],
        "active_volume": [av_tong["active_buy_volume"], av_tong["active_sell_volume"], np.nan],
        "từ tick": [
            tick[tick["side"] == "buy"]["volume"].sum(),
            tick[tick["side"] == "sell"]["volume"].sum(),
            tick[tick["side"] == "auction"]["volume"].sum(),
        ],
    },
    index=["mua chủ động", "bán chủ động", "định kỳ"],
)
so_sanh.loc["TỔNG"] = so_sanh.sum()
so_sanh.round(0)

,net_active_value,active_volume,từ tick
mua chủ động,7233400.0,7817150.0,7233400.0
bán chủ động,8331800.0,8915550.0,8331800.0
định kỳ,1167500.0,NaN,1167500.0
TỔNG,16732700.0,16732700.0,16732700.0


Ba cột cùng có **TỔNG bằng khối lượng khớp cả phiên**, nhưng cột giữa chỉ có
hai dòng. Vậy `active_volume` giấu phần định kỳ vào đâu?

In [11]:
lech_mua = av_tong["active_buy_volume"] - nav_tong["buy_volume"]
lech_ban = av_tong["active_sell_volume"] - nav_tong["sell_volume"]

print(f"active_buy_volume  − buy_volume  = {lech_mua:>12,.0f}")
print(f"active_sell_volume − sell_volume = {lech_ban:>12,.0f}")
print(f"auction_volume ÷ 2               = {nav_tong['auction_volume'] / 2:>12,.0f}")
print(f"\n→ `active_volume` CHIA ĐÔI khối lượng định kỳ vào hai rổ mua và bán.")

active_buy_volume  − buy_volume  =      583,750
active_sell_volume − sell_volume =      583,750
auction_volume ÷ 2               =      583,750

→ `active_volume` CHIA ĐÔI khối lượng định kỳ vào hai rổ mua và bán.


### Kiểm chứng trên nhiều mã, nhiều phiên

Một lần trùng có thể là ngẫu nhiên. Đo trên ba mã × năm phiên:

In [12]:
KIEM = ["HPG", "VCB", "SSI"]
bat_dau = (PHIEN - pd.Timedelta(days=8)).strftime("%Y-%m-%d")

dong = []
for ma in KIEM:
    n = client.intraday.stock.net_active_value(ma, interval="1h", start=bat_dau, end=ngay_str)
    a = client.eod.stock.active_volume(ma, start=bat_dau, end=ngay_str)
    n_ngay = n.assign(d=n["time"].dt.date).groupby("d")[["buy_volume", "sell_volume", "auction_volume"]].sum()
    a_ngay = a.assign(d=a["date"].dt.date).set_index("d")[["active_buy_volume", "active_sell_volume"]]
    ghep = n_ngay.join(a_ngay, how="inner")
    ghep["lệch mua"] = ghep["active_buy_volume"] - ghep["buy_volume"]
    ghep["định kỳ ÷ 2"] = ghep["auction_volume"] / 2
    ghep["khớp"] = (ghep["lệch mua"] - ghep["định kỳ ÷ 2"]).abs() < 1
    dong.append(ghep.assign(mã=ma))

kiem_chung = pd.concat(dong)
print(f"Giả thuyết 'chia đôi khối lượng định kỳ' đúng ở "
      f"{kiem_chung['khớp'].sum()}/{len(kiem_chung)} quan sát ({kiem_chung['khớp'].mean():.0%})")
kiem_chung[["mã", "buy_volume", "auction_volume", "active_buy_volume", "lệch mua", "định kỳ ÷ 2", "khớp"]].round(0)

Giả thuyết 'chia đôi khối lượng định kỳ' đúng ở 18/18 quan sát (100%)


,mã,buy_volume,auction_volume,active_buy_volume,lệch mua,định kỳ ÷ 2,khớp
d,,,,,,,
2026-08-03,HPG,28242300.0,2041300.0,29262950.0,1020650.0,1020650.0,True
2026-08-04,HPG,8484900.0,2146300.0,9558050.0,1073150.0,1073150.0,True
2026-08-05,HPG,8012500.0,2071300.0,9048150.0,1035650.0,1035650.0,True
2026-08-06,HPG,5193100.0,2209600.0,6297900.0,1104800.0,1104800.0,True
2026-08-07,HPG,6355600.0,1389900.0,7050550.0,694950.0,694950.0,True
2026-08-10,HPG,7233400.0,1167500.0,7817150.0,583750.0,583750.0,True
2026-08-03,VCB,5338400.0,168300.0,5422550.0,84150.0,84150.0,True
2026-08-04,VCB,2041300.0,400900.0,2241750.0,200450.0,200450.0,True
2026-08-05,VCB,1304200.0,226300.0,1417350.0,113150.0,113150.0,True


### Hệ quả thực hành

| Bạn muốn | Dùng | Vì sao |
|---|---|---|
| Mua/bán **chủ động** thật, không lẫn định kỳ | `intraday.*.net_active_value()` | ba rổ tách bạch |
| Chỉ cần **chênh lệch ròng** mua − bán | cả hai đều được | phần định kỳ chia đôi nên nó triệt tiêu khi trừ |
| Tỷ lệ mua chủ động trên tổng | **phải** dùng `net_active_value` | `active_volume` thổi phồng tử số bằng nửa phần định kỳ |
| Giá trị bằng VND | **chỉ** `net_active_value` | `active_volume` không có cột tiền nào |

In [13]:
print("Chênh lệch ròng — hai nguồn cho cùng một số:")
print(f"  net_active_value : {nav_tong['buy_volume'] - nav_tong['sell_volume']:>12,.0f}")
print(f"  active_volume    : {av_tong['active_buy_volume'] - av_tong['active_sell_volume']:>12,.0f}")
print()
print("Nhưng tỷ lệ mua chủ động thì KHÔNG:")
print(f"  net_active_value : {nav_tong['buy_volume'] / KL_EOD:.2%}  ← đúng")
print(f"  active_volume    : {av_tong['active_buy_volume'] / KL_EOD:.2%}  ← thổi phồng")

Chênh lệch ròng — hai nguồn cho cùng một số:
  net_active_value :   -1,098,400
  active_volume    :   -1,098,400

Nhưng tỷ lệ mua chủ động thì KHÔNG:
  net_active_value : 43.23%  ← đúng
  active_volume    : 46.72%  ← thổi phồng


## 6 · `supply_demand()` — lệnh ĐẶT, không phải lệnh KHỚP

Đây là bảng dễ bị dùng nhầm nhất, vì tên cột nghe giống khối lượng giao dịch.
**Mọi cột đều mang chữ `_order_` chính vì lý do đó.**

In [14]:
sd_hang = sd.iloc[0]
print(f"Khối lượng ĐẶT mua : {sd_hang['buy_order_volume']:>14,.0f}")
print(f"Khối lượng ĐẶT bán : {sd_hang['sell_order_volume']:>14,.0f}")
print(f"Khối lượng KHỚP    : {KL_EOD:>14,.0f}")
print(f"\nTỷ lệ đặt mua / khớp: {sd_hang['buy_order_volume'] / KL_EOD:.2f} lần")

Khối lượng ĐẶT mua :     28,695,984
Khối lượng ĐẶT bán :     35,014,584
Khối lượng KHỚP    :     16,732,700

Tỷ lệ đặt mua / khớp: 1.71 lần


Lệnh đặt lớn hơn lệnh khớp nhiều lần — phần lớn lệnh vào sổ rồi bị huỷ hoặc
không tới lượt khớp. **Hai bảng này không trừ được cho nhau và không trừ được
cho `ohlcv()`.**

In [15]:
# Ba cột `_count` có kiểu Int64 **nullable** — chúng CÓ THỂ rỗng, khác với các
# cột float. Đo trên một rổ mã để thấy null có thật:
RO_KIEM = ["HPG", "VCB", "SSI", "FPT", "MWG", "STB", "VIX", "SHB"]
lich_sd = client.eod.stock.supply_demand(RO_KIEM, start=lui_ngay(HOM_NAY, nam=1))

print(f"kiểu dữ liệu của buy_order_count: {lich_sd['buy_order_count'].dtype}")
print(f"số dòng null: {lich_sd['buy_order_count'].isna().sum():,}/{len(lich_sd):,} "
      f"({lich_sd['buy_order_count'].isna().mean():.2%})")
theo_ma_null = lich_sd.groupby("symbol", observed=True)["buy_order_count"].apply(lambda s: s.isna().mean())
print("\nTỷ lệ null theo mã:")
print((theo_ma_null * 100).round(1).astype(str).add("%").to_string())
print("\n⚠️ Kiểm bằng `.isna()`, đừng so với 0 — null nghĩa là 'không có số liệu',")
print("   còn 0 nghĩa là 'không có lệnh nào'. Hai chuyện khác nhau, và với kiểu")
print("   Int64 nullable thì `df[df.buy_order_count == 0]` bỏ sót toàn bộ nhóm null.")

kiểu dữ liệu của buy_order_count: Int64
số dòng null: 0/1,992 (0.00%)

Tỷ lệ null theo mã:
symbol
FPT    0.0%
HPG    0.0%
MWG    0.0%
SHB    0.0%
SSI    0.0%
STB    0.0%
VCB    0.0%
VIX    0.0%

⚠️ Kiểm bằng `.isna()`, đừng so với 0 — null nghĩa là 'không có số liệu',
   còn 0 nghĩa là 'không có lệnh nào'. Hai chuyện khác nhau, và với kiểu
   Int64 nullable thì `df[df.buy_order_count == 0]` bỏ sót toàn bộ nhóm null.


## 7 · ⚠️ `max_intraday_days` — ràng buộc thật của Track này

In [16]:
han = client.limits()
print(f"max_intraday_days = {han['max_intraday_days']}")

try:
    client.intraday.stock.ohlcv(
        MA, interval="1h", start=lui_ngay(HOM_NAY, thang=3), end=PHIEN.strftime("%Y-%m-%d"), on_error="raise"
    )
    print("→ Lấy được 3 tháng trong một lời gọi")
except finlens.FinLensError as e:
    print(f"→ {type(e).__name__}: {e}")

max_intraday_days = 10


→ InvalidDateRangeError: [FL_VALIDATION_DATE_RANGE] Khoảng bạn yêu cầu dài 90 ngày, nhưng dữ liệu trong phiên chỉ lấy được tối đa 10 ngày mỗi request. Hãy chia thành nhiều request nhỏ hơn, hoặc nâng cấp gói. Con số này nằm trong `meta.limits.max_intraday_days` của mọi response. (request_id=48eb0a138559412db44ca50106cec616) -> https://docs.finlens.vn/python-sdk/errors/FL_VALIDATION_DATE_RANGE


Cách sống chung: lặp theo cửa sổ. Hàm dưới đây là mẫu bạn sẽ dùng lại mỗi khi
cần chuỗi intraday dài.

In [17]:
def tai_intraday(ma: str, *, tu: str, den: str, interval: str = "1h") -> pd.DataFrame:
    """Tải intraday qua nhiều cửa sổ ≤ max_intraday_days rồi ghép lại.

    ⚠️ `attrs` không sống sót qua `pd.concat` — đọc đơn vị ở cửa sổ đầu và gắn
    lại thủ công.
    """
    buoc = client.limits()["max_intraday_days"]
    # ⚠️ `start` và `end` đều TÍNH VÀO khoảng, nên hai mốc cách nhau `buoc` ngày
    # tạo ra một cửa sổ dài `buoc + 1` ngày và bị từ chối. Bước phải là `buoc - 1`.
    moc = pd.date_range(tu, den, freq=f"{buoc - 1}D").tolist()
    if pd.Timestamp(den) not in moc:
        moc.append(pd.Timestamp(den))

    phan, don_vi = [], None
    for a, b in zip(moc[:-1], moc[1:], strict=True):
        d = client.intraday.stock.ohlcv(
            ma, interval=interval, start=a.strftime("%Y-%m-%d"), end=b.strftime("%Y-%m-%d")
        )
        if d.empty:
            continue
        don_vi = don_vi or d.attrs["finlens"]["units"]
        phan.append(d)

    ghep = pd.concat(phan, ignore_index=True).drop_duplicates(subset=["symbol", "time"])
    ghep.attrs["finlens"] = {"units": don_vi}
    return ghep.sort_values("time")


ba_thang = tai_intraday(MA, tu=lui_ngay(HOM_NAY, thang=3), den=PHIEN.strftime("%Y-%m-%d"))
print(f"{len(ba_thang):,} thanh 1 giờ · {ba_thang['time'].min():%d/%m/%Y} → {ba_thang['time'].max():%d/%m/%Y}")
print(f"Đơn vị giá: {ba_thang.attrs['finlens']['units']['close']}")

319 thanh 1 giờ · 13/05/2026 → 10/08/2026
Đơn vị giá: kVND


Hai chi tiết nhỏ trong hàm trên, cả hai đều là lỗi thật đã gặp khi viết
notebook này:

1. **Bước lặp là `buoc - 1`, không phải `buoc`.** `start` và `end` đều tính vào
   khoảng, nên hai mốc cách nhau đúng 10 ngày tạo ra một cửa sổ **11 ngày** và
   server từ chối với `InvalidDateRangeError`.
2. **`drop_duplicates` không phải thừa.** Hai cửa sổ liền kề chia sẻ ngày biên,
   nên không có nó thì ngày đó xuất hiện hai lần và mọi phép tổng đều lệch.

## 8 · Dùng thật: dòng tiền chủ động theo giờ, ba tháng

Câu hỏi có ích: trong phiên, khung giờ nào thường có áp lực mua và khung nào có
áp lực bán?

In [18]:
nav_dai = []
buoc = han["max_intraday_days"]
moc = pd.date_range(lui_ngay(HOM_NAY, thang=3), PHIEN, freq=f"{buoc - 1}D").tolist() + [PHIEN]
for a, b in zip(moc[:-1], moc[1:], strict=True):
    d = client.intraday.stock.net_active_value(
        MA, interval="1h", start=a.strftime("%Y-%m-%d"), end=b.strftime("%Y-%m-%d")
    )
    if not d.empty:
        nav_dai.append(d)

nav_dai = pd.concat(nav_dai, ignore_index=True).drop_duplicates(subset=["symbol", "time"])
print(f"{len(nav_dai):,} thanh giờ")

theo_gio = (
    nav_dai.assign(gio=nav_dai["time"].dt.hour)
    .groupby("gio", observed=True)
    .agg(mua_rong_ty=("net_value", lambda s: s.sum() / 1e9), so_thanh=("net_value", "count"))
    .reset_index()
)
theo_gio["gio"] = theo_gio["gio"].astype(str) + "h"

from finlens_examples import thanh_doi_mau

thanh_doi_mau(
    theo_gio,
    x="gio",
    y="mua_rong_ty",
    tieu_de=f"{MA} — mua/bán chủ động ròng theo khung giờ, 3 tháng",
    phu_de="Cộng dồn giá trị ròng của từng khung giờ qua toàn bộ số phiên",
    nhan_y="tỷ đồng",
    dinh_dang_nhan="{:+,.0f}",
)

319 thanh giờ


## 9 · Xuất báo cáo phiên ra Excel

⚠️ **Excel không nhận datetime có múi giờ.** Cột `time` của intraday mang
`+07:00`, nên `to_excel` ném `ValueError`. Phải bỏ thông tin múi giờ trước khi
ghi — và bỏ bằng `tz_localize(None)` (giữ nguyên giờ Việt Nam) chứ **không**
bằng `tz_convert("UTC")`, vốn dời mọi mốc lùi 7 tiếng và làm phiên sáng nhảy
sang ngày hôm trước.

In [19]:
THU_MUC_RA = GOC / "output"
THU_MUC_RA.mkdir(exist_ok=True)
tep = THU_MUC_RA / f"phan_tich_intraday_{MA}_{PHIEN:%Y%m%d}.xlsx"

tick_xuat = tick[["time", "price", "volume", "side", "value"]].head(5000).copy()
tick_xuat["time"] = tick_xuat["time"].dt.tz_localize(None)

phut_xuat = phut.copy()
phut_xuat.index = phut_xuat.index.tz_localize(None)

with pd.ExcelWriter(tep) as w:
    theo_side.to_excel(w, sheet_name="Co cau side")
    so_sanh.round(0).to_excel(w, sheet_name="Ba dai luong")
    phut_xuat.to_excel(w, sheet_name="Dong lenh 5 phut")
    tick_xuat.to_excel(w, sheet_name="Tick 5000 dau", index=False)

print(f"Đã ghi 4 sheet → {tep.name} ({tep.stat().st_size / 1024:,.0f} KB)")

Đã ghi 4 sheet → phan_tich_intraday_HPG_20260810.xlsx (97 KB)


## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Từng lệnh khớp một phiên | `intraday.stock.ticks("HPG", date="...")` |
| Mua/bán chủ động **tách định kỳ** | `intraday.stock.net_active_value()` |
| Chênh lệch ròng theo phiên | `eod.stock.active_volume()` cũng được |
| Lệnh **đặt** vào sổ | `eod.stock.supply_demand()` |
| Chuỗi intraday dài | lặp theo cửa sổ ≤ `max_intraday_days` |

**Bốn điều mang sang notebook sau:**

1. `side` có **ba** giá trị. Phiên định kỳ chiếm hai chữ số phần trăm khối
   lượng ở nhiều mã — không giấu được vào rổ mua hay rổ bán.
2. **`active_volume` chia đôi khối lượng định kỳ vào hai rổ.** Chênh lệch ròng
   vẫn đúng (phần chia đôi triệt tiêu), nhưng **tỷ lệ mua chủ động thì sai**.
   Kiểm chứng trong notebook này khớp 15/15 quan sát.
3. `supply_demand()` là lệnh **đặt**, gấp nhiều lần lệnh khớp. Không trừ được
   cho `ohlcv()`. Ba cột `_count` là nullable — kiểm bằng `.isna()`.
4. `max_intraday_days` là ràng buộc thật. Lặp theo cửa sổ, và **`drop_duplicates`
   ở ngày biên**.
5. Bốn bề mặt EOD **không cùng độ trễ** — sổ lệnh đặt về chậm hơn giá một
   phiên. Chọn phiên phân tích bằng `min()` các mốc cuối, không phải bằng mốc
   cuối của một bảng.
6. Cột `time` của intraday mang múi giờ `+07:00`. Excel không nhận nó — dùng
   `tz_localize(None)`, không phải `tz_convert("UTC")`.

---

**Tiếp theo:** [`42_phai_sinh_va_basis.ipynb`](42_phai_sinh_va_basis.ipynb) —
VN30F1M, chênh lệch với chỉ số cơ sở, và ba đơn vị cùng lúc.